# Nonlinear machine-learning sensitivity comparisons

**Purpose.** This notebook preserves the validated nonlinear comparisons and places them beside the final year-controlled principal sequence. No model, feature or hyperparameter search is reopened.

**Principal inputs.** Frozen nonlinear predictions and tuning selections; clean Y52 cohort; frozen folds; `supervisor_confirmed_update_v1` metrics.

**Main outputs.** Exact reproduction of selected Extra Trees and histogram-gradient-boosting predictions, historical Random Forest evidence, and a clearly separated comparison with the final principal models.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Find the repository from the current working directory so the notebook
# remains portable when the project folder is moved or opened in Colab.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = next((p for p in candidates if (p / "scripts/thesis_utils_v2.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the Spotify_clv project.")
sys.path.insert(0, str(ROOT / "scripts"))

from thesis_utils_v2 import (
    SEED, assert_close, assert_unique_tracks, blocked_predictor_hits,
    concise_table, environment_table, extract_spotify_id, load_json,
    read_chart_archive, record_values, regression_metrics,
    save_and_show_figure, setup_notebook,
)

ROOT, FIGURE_DIR, RECORD_DIR = setup_notebook("05_nonlinear_ml_comparisons.ipynb")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(environment_table())
print(f"Random seed: {SEED}")


## Preserved scope and fair comparison

Random Forest, Extra Trees and histogram gradient boosting remain nonlinear sensitivity models. The selected Extra Trees and gradient-boosting outer fits are reproduced with their frozen configurations and the same 1,014 tracks and protected folds. Entry year is not retrofitted into these sensitivity models because the professor requested a control in the principal sequence, not a new nonlinear search.


In [ ]:
import run_provisional_final_model_comparison as final_models

audit_rows = []
data, data26, assignments, temporal52, temporal26 = final_models.validate_inputs(audit_rows)
tuning = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/ml_inner_grouped_tuning.csv"
)
specifications = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/ml_model_specifications.csv"
)
display(specifications)
selected = tuning[tuning.selected.eq(1)][[
    "context", "algorithm", "feature_block", "config_id", "inner_mean_mae", "outer_fold"
]]
display(selected.round(4))
assert len(data) == 1014
assert assignments.primary_group_component.nunique() == 276
assert blocked_predictor_hits(final_models.F3 + final_models.LEX) == []


## Refit the selected outer-fold models

The exhaustive inner calculations are not repeated. Instead, each outer model is refitted with the configuration already selected from its training data. This reproduces held-out predictions without using the outer outcomes to retune the model.


In [ ]:
historical_ml = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/ml_grouped_cv_predictions.csv"
)
config_lookup = {
    "ET": {item["config_id"]: item for item in final_models.ET_CONFIGS},
    "HGB": {item["config_id"]: item for item in final_models.HGB_CONFIGS},
}
selected_models = [
    ("ET", "ML_F3", "ET_ML_F3"),
    ("HGB", "ML_LEX", "HGB_ML_LEX"),
]
reproduced_parts = []
for fold in sorted(data.fold.unique()):
    train = data[data.fold.ne(fold)].copy()
    heldout = data[data.fold.eq(fold)].copy()
    for algorithm, block, model_id in selected_models:
        historical_part = historical_ml[
            historical_ml.fold.eq(fold) & historical_ml.model_id.eq(model_id)
        ]
        config_id = historical_part.selected_config_id.unique().tolist()
        assert len(config_id) == 1
        config = config_lookup[algorithm][config_id[0]]
        variables = final_models.BLOCKS[block]
        # Tree models do not need scaling; encoders are nevertheless fitted on training only.
        pipeline = final_models.make_tree_pipeline(
            algorithm, config, variables, SEED + int(fold.split("_")[-1])
        )
        pipeline.fit(train[variables], train.Y_52)
        prediction = pipeline.predict(heldout[variables])
        reproduced_parts.append(pd.DataFrame({
            "spotify_track_id": heldout.spotify_track_id,
            "fold": fold, "model_id": model_id,
            "y_observed": heldout.Y_52, "y_pred_raw": prediction,
            "X": heldout.X,
        }))
reproduced_ml = pd.concat(reproduced_parts, ignore_index=True)
assert not reproduced_ml.duplicated(["spotify_track_id", "model_id"]).any()

aligned = reproduced_ml.merge(
    historical_ml[["spotify_track_id", "model_id", "y_pred_raw"]].rename(
        columns={"y_pred_raw": "validated_prediction"}
    ),
    on=["spotify_track_id", "model_id"], validate="one_to_one"
)
max_prediction_difference = (aligned.y_pred_raw - aligned.validated_prediction).abs().max()
assert max_prediction_difference < 1e-10

reproduced_metrics = pd.DataFrame([
    {"model_id": model_id, **regression_metrics(part.y_observed, part.y_pred_raw)}
    for model_id, part in reproduced_ml.groupby("model_id")
])
display(reproduced_metrics.round(3))
print("Maximum difference from frozen held-out prediction:", max_prediction_difference)


## Comparison with the final principal sequence

The nonlinear metrics below are preserved pre-update sensitivity evidence. F3_YEAR and LEX_YEAR are shown from the targeted supervisor update so the thesis-facing comparison uses the correct final reference while keeping the nonlinear experiments unchanged.


In [ ]:
final_comparison = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/thesis_final_model_comparison.csv"
)
year_metrics = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_grouped_cv_metrics.csv"
)
year_rows = year_metrics[year_metrics.model_id.isin(["F3_YEAR", "LEX_YEAR"])].rename(columns={
    "mae": "grouped_cv_mae", "rmse": "grouped_cv_rmse",
    "mean_prediction_bias": "grouped_cv_bias",
})[["model_id", "grouped_cv_mae", "grouped_cv_rmse", "grouped_cv_bias"]]
historical_rows = final_comparison[
    final_comparison.model_id.isin(["RF_F3", "ET_ML_F3", "HGB_ML_LEX"])
][["model_id", "grouped_cv_mae", "grouped_cv_rmse", "grouped_cv_bias"]]
comparison = pd.concat([year_rows, historical_rows], ignore_index=True)
display(comparison.round(3))

plot = comparison.melt(
    id_vars=["model_id"], value_vars=["grouped_cv_mae", "grouped_cv_rmse"],
    var_name="metric", value_name="error"
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=plot, x="model_id", y="error", hue="metric",
            palette=["#4C78A8", "#F58518"], ax=ax)
ax.set(title="Final transparent models and preserved nonlinear sensitivities",
       xlabel="Model", ylabel="Prediction error (chart weeks)")
ax.legend(title="", labels=["MAE", "RMSE"])
save_and_show_figure(fig, FIGURE_DIR / "nb05_nonlinear_vs_year_controlled.png")


Neither nonlinear sensitivity changes the conclusion drawn from the final year-controlled sequence. Extra Trees remains near F3_YEAR but does not improve on LEX_YEAR, while histogram gradient boosting is worse on both primary error measures. These are retained comparisons, not candidates reopened after the supervisor's request.


In [ ]:
ml_uncertainty = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/ml_paired_cluster_bootstrap.csv"
)
display(ml_uncertainty[ml_uncertainty.metric.isin(["MAE", "RMSE"])][[
    "reference_model", "candidate_model", "metric", "observed_difference",
    "ci_2_5", "ci_97_5", "bootstrap_proportion_candidate_better"
]].round(4))

temporal = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/ml_temporal_metrics_52w.csv"
)
display(temporal[[
    "temporal_partition", "model_id", "n", "mae", "rmse",
    "mean_prediction_bias", "outside_bound_n"
]].round(3))

intervals = ml_uncertainty[
    ml_uncertainty.metric.eq("MAE") & ml_uncertainty.reference_model.eq("LEX")
].copy()
intervals["lower_error"] = intervals.observed_difference
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.errorbar(
    intervals.observed_difference, intervals.candidate_model,
    xerr=[intervals.observed_difference - intervals.ci_2_5,
          intervals.ci_97_5 - intervals.observed_difference],
    fmt="o", color="#4C78A8", capsize=4
)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set(title="Candidate MAE improvement over LEX", xlabel="LEX MAE minus candidate MAE (weeks)", ylabel="")
save_and_show_figure(fig, FIGURE_DIR / "nb05_ml_cluster_intervals.png")


The historical paired intervals did not support promotion of either new tree method over the transparent lexical model. Because the entry-year update changes the transparent errors only minimally, no targeted result justifies rerunning nonlinear tuning or uncertainty analysis.


In [ ]:
importance = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/ml_feature_importance.csv"
)
joint = importance[
    importance.aggregation.eq("weighted_across_5_outer_folds")
    & importance.importance_scope.eq("joint_conceptual_group")
    & (
        (importance.algorithm.eq("ET") & importance.feature_block.eq("ML_F3"))
        | (importance.algorithm.eq("HGB") & importance.feature_block.eq("ML_LEX"))
    )
].copy()
display(joint[["algorithm", "feature_block", "conceptual_group", "mae_increase_mean", "mae_increase_sd"]].round(3))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=joint, x="conceptual_group", y="mae_increase_mean", hue="algorithm",
            palette=["#4C78A8", "#E45756"], ax=ax)
ax.set(title="Held-out joint permutation importance", xlabel="Predictor group", ylabel="Increase in MAE after permutation (weeks)")
save_and_show_figure(fig, FIGURE_DIR / "nb05_ml_group_importance.png")

bounds = reproduced_metrics[["model_id", "outside_support_n"]].copy()
display(bounds)


X dominates both nonlinear sensitivities. Audio and lexical groups contribute little additional permutation importance once X is present. Extra Trees stays inside the outcome support; histogram gradient boosting retains the documented negative-prediction issue. No clipping is applied to primary metrics.

## Notebook 05 conclusions

The preserved nonlinear evidence does not alter the thesis hierarchy. It supports retaining simple, interpretable year-controlled count models in the main results and placing nonlinear comparisons in sensitivity material.


In [ ]:
metric_index = reproduced_metrics.set_index("model_id")
rf = final_comparison.set_index("model_id").loc["RF_F3"]
record_values(RECORD_DIR / "nb05_values.json", "05_nonlinear_ml_comparisons", {
    "et_mae": metric_index.loc["ET_ML_F3", "mae"],
    "et_rmse": metric_index.loc["ET_ML_F3", "rmse"],
    "hgb_mae": metric_index.loc["HGB_ML_LEX", "mae"],
    "hgb_rmse": metric_index.loc["HGB_ML_LEX", "rmse"],
    "hgb_outside_support_n": metric_index.loc["HGB_ML_LEX", "outside_support_n"],
    "rf_mae": rf.grouped_cv_mae,
    "rf_rmse": rf.grouped_cv_rmse,
    "max_reproduction_difference": max_prediction_difference,
})
print("Notebook 05 reconciliation record saved.")
